In [ ]:
library(MLmetrics)
library(randomForest)
library(dplyr)
set.seed(2) 


ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

In [ ]:
datam<-read.csv("data_target_encoding.csv",stringsAsFactors = T)

In [ ]:
n_trees <- c(10,20,100,200,500)
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)),floor(sqrt(nfeat)),floor(2*sqrt(nfeat)),nfeat)
accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
id_variable <- match('building_id', colnames(datam))
target_variable <- match('damage_grade', colnames(datam))
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

for (i in n_trees){ 
    pb2 <- txtProgressBar(min = 0, max = length(m_tries), style = 3)
    for (j in m_tries){
        #3.1 Take the first half of the dataset as a training data set
        print(paste("Number of trees used:",i,'mTry:',j))
        train_data <- datam[datam_idx[1:split],-c(id_variable)]

        #3.2 Take the second half of the dataset as a hold out or test data set
        test_data <- datam[datam_idx[(split+1):nrows],-c(id_variable)]
        
        model <- randomForest(x=train_data[,-c(target_variable)],
                              y=as.factor(train_data[,c(target_variable)]),
                              ntree=i,mtry=j,doBest=TRUE,keep.forest=TRUE,importance=TRUE,do.trace=TRUE)
        #model<-tuneRF(x=train_data[,-c(target_variable),drop=F],
        #    y=as.factor(train_data[,c(target_variable)]),ntreeTry=i,mtryStart=12,plot=TRUE,trace=TRUE,doBest=TRUE,importance=TRUE,do.trace=TRUE)
        #model
        yhat<-predict(model,test_data[,-c(target_variable),drop=F])                      
        accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
        print(paste("Best F1 Score - ",i, 'trees','- mtry',j,':',accuracy_vec[nrow(accuracy_vec),3],n=1))
        setTxtProgressBar(pb2, j)
        rm('model')
    }
    setTxtProgressBar(pb, i)
}
print(accuracy_vec)
#write.csv(accuracy_vec,'randomForest_tuning.csv') 




In [ ]:
k = 10

accuracy_vec <- array(0,k)

n_trees <- c(10,20,100,200,500)
nfeat <- ncol(datam)-2 #I remove 2 to remove building_id and damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)),floor(sqrt(nfeat)),floor(2*sqrt(nfeat)),nfeat)
accuracy_vec <- data.frame(matrix(ncol = 4, nrow = 0))
colnames(accuracy_vec)<-c('fold','n_trees','mtry','F1')
nrows<-nrow(datam)
datam_idx <- sample(1:nrows)
split <- floor(nrows*0.8)
id_variable <- match('building_id', colnames(datam))
target_variable <- match('damage_grade', colnames(datam))
pb <- txtProgressBar(min = 0, max = length(n_trees), style = 3)

# 1. Shuffle the dataset randomly.
datam_idx <- sample(1:nrow(datam))

# 2. Split the dataset into k groups
max <- ceiling(nrow(datam)/k)
splits <- split(datam_idx, ceiling(seq_along(datam_idx)/max))

pb <- txtProgressBar(min = 0, max = k*length(n_trees)*length(m_tries), style = 3)
counter <-0

# 3. For each unique group:
for (i in 1:k){
    for (j in (n_trees)){
        for (l in m_tries){
            #3.1 Take the group as a hold out or test data set
            test_data <- datam[splits[[i]],-id_variable]


            #3.2 Take the remaining groups as a training data set
            train_data <- datam[-splits[[i]],-id_variable]   

            model <- randomForest(x=train_data[,-c(target_variable)],
                                y=as.factor(train_data[,c(target_variable)]),
                                ntree=j,mtry=l,keep.forest=TRUE,importance=TRUE)
            yhat<-predict(model,test_data[,-c(target_variable)])
            accuracy_vec[nrow(accuracy_vec)+1,]<-c(i,j,l,F1_Score_micro(as.factor(test_data[,c(target_variable)]),yhat))
            counter<-counter+1
            setTxtProgressBar(pb, counter)
            print(paste("Best F1 Score - ",i, 'fold',j,'trees',l,'mtry',':',accuracy_vec[nrow(accuracy_vec),4],n=1))
            rm('model')
        }
    }
}

#4. Summarize the accuracy of the model using the sample of model evaluation scores
mean_F1 <- accuracy_vec %>% group_by(fold) %>% 
  summarise(mean_F1=mean(F1),
            .groups = 'drop')
mean_F1